# Findings detector — DentalOPG 1550

Covers what the DENTEX disease detector does not: **missing teeth**, broken-down
roots (BDR), fillings, crowns, root canals, bridges, impacted/unerupted teeth,
periapical abscess and periodontitis.

DentalOPG: 1,068 train / 303 valid / 154 test, 640x640, YOLO labels.

Two source classes are dropped:

- `Caries` — already covered by the DENTEX disease detector; keeping it would
  double-report the same lesion under two taxonomies.
- `Normal Teeth` (5,076 boxes) — a dominant "every tooth" class that would
  teach the model to box normal dentition instead of findings.
- `Mixed Dentition` (3 boxes) — too thin to learn.

The remaining 11 classes are trained with 3x oversampling for the thin ones
(Bridge, Periodontitis, Distoangular, Apical Abscess).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# ultralytics + pretrained weights come from the private wheels dataset.
WHEELS = next(Path("/kaggle/input").rglob("yolov8s.pt")).parent
os.environ["YOLO_OFFLINE"] = "True"

try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                    "--find-links", str(WHEELS), "ultralytics", "ultralytics-thop"],
                   check=True)
    import ultralytics

import torch
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: enable GPU + phone verification on Kaggle.")

SRC = next(p.parent for p in Path("/kaggle/input").rglob("data.yaml")
           if (p.parent / "train" / "images").exists())
print("dataset:", SRC)


In [ ]:
import json
import shutil
from collections import Counter
from pathlib import Path

WORK = Path("/kaggle/working")
DS = WORK / "ds"

# original DentalOPG id -> new id; None = drop
MAP = {0: 0, 1: 1, 2: 2, 3: None, 4: 3, 5: 4, 6: 5, 7: 6,
       8: None, 9: None, 10: 7, 11: 8, 12: 9, 13: 10}
NAMES = ["BDR", "Bridge", "Cap", "Distoangular Impaction", "Filling",
         "Horizontal Impaction", "Missing Teeth", "Periodontitis",
         "Root Canal", "Unerupted Wisdom", "Apical Abscess"]

counts = {}
for split in ("train", "valid"):
    (DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (DS / "labels" / split).mkdir(parents=True, exist_ok=True)
    kept = Counter()
    n_img = 0
    for img in sorted((SRC / split / "images").iterdir()):
        lab = SRC / split / "labels" / (img.stem + ".txt")
        if not lab.exists():
            continue
        rows = []
        for line in lab.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            new = MAP[int(parts[0])]
            if new is None:
                continue
            rows.append(f"{new} " + " ".join(parts[1:]))
            kept[new] += 1
        shutil.copy2(img, DS / "images" / split / img.name)
        (DS / "labels" / split / (img.stem + ".txt")).write_text(
            "\n".join(rows) + ("\n" if rows else ""))
        n_img += 1
    counts[split] = kept
    print(split, n_img, "images |", dict(sorted(kept.items())))

assert len(list((DS / "images" / "train").iterdir())) == 1068
assert len(list((DS / "images" / "valid").iterdir())) == 303
assert counts["train"][6] == 1087 and counts["train"][0] == 563
print("class boxes (train):", {NAMES[i]: counts["train"][i] for i in range(len(NAMES))})


In [ ]:
import yaml

THIN = {1, 3, 7, 10}  # Bridge, Distoangular, Periodontitis, Apical Abscess
lines = []
for img in sorted((DS / "images" / "train").iterdir()):
    lines.append(str(img))
    lab = DS / "labels" / "train" / (img.stem + ".txt")
    classes = {int(l.split()[0]) for l in lab.read_text().splitlines() if l.strip()}
    if classes & THIN:
        lines += [str(img)] * 2  # 3x total

(DS / "train.txt").write_text("\n".join(lines) + "\n")
(DS / "val.txt").write_text(
    "\n".join(str(p) for p in sorted((DS / "images" / "val").iterdir())) + "\n")

data = {"train": str(DS / "train.txt"), "val": str(DS / "val.txt"),
        "names": {i: n for i, n in enumerate(NAMES)}}
(DS / "data.yaml").write_text(yaml.safe_dump(data, sort_keys=False))
print("train list:", len(lines), "| val:", len((DS / "val.txt").read_text().splitlines()))


In [ ]:
from ultralytics import YOLO

IMGSZ, EPOCHS, BATCH = 640, 40, 16
model = YOLO(str(WHEELS / "yolov8s.pt"))
results = model.train(
    data=str(DS / "data.yaml"), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
    patience=10, seed=42, deterministic=True, optimizer="auto", cos_lr=True,
    close_mosaic=10, amp=True, workers=2, project=str(WORK / "runs"),
    name="findings", exist_ok=True, fliplr=0.5, flipud=0.0, mosaic=0.5,
    degrees=5.0, translate=0.1, scale=0.3, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
)


In [ ]:
from pathlib import Path
from ultralytics import YOLO

best_model = YOLO(str(Path(results.save_dir) / "weights" / "best.pt"))
m = best_model.val(data=str(DS / "data.yaml"), imgsz=IMGSZ, split="val", conf=0.001, iou=0.6)
names = [m.names[i] for i in sorted(m.names)]
ap50 = m.box.ap50.tolist()

print(f"mAP50    {m.box.map50:.4f}")
print(f"mAP50-95 {m.box.map:.4f}")
print(f"precision {m.box.mp:.4f} | recall {m.box.mr:.4f}")
print()
for name, ap in sorted(zip(names, ap50), key=lambda t: -t[1]):
    print(f"  {name:24s} AP50 {ap:.3f}")

summary = {
    "model": str(WHEELS / "yolov8s.pt"), "imgsz": IMGSZ, "epochs": EPOCHS,
    "batch": BATCH, "train_images": len(lines), "val_images": 303,
    "mAP50": m.box.map50, "mAP50_95": m.box.map,
    "precision": m.box.mp, "recall": m.box.mr,
    "ap50_per_class": dict(zip(names, ap50)), "classes": NAMES,
}
(WORK / "findings_metrics.json").write_text(json.dumps(summary, indent=2))
print("wrote findings_metrics.json")


In [ ]:
best = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best, WORK / "findings_best.pt")
shutil.make_archive(str(WORK / "findings_run"), "zip", str(results.save_dir))
print("findings_best.pt:", (WORK / "findings_best.pt").stat().st_size / 1e6, "MB")

# keep the packaged kernel output small: images and run tree are not artifacts
for p in (DS, WORK / "runs"):
    shutil.rmtree(p, ignore_errors=True)
print("kernel output:", sorted(x.name for x in WORK.iterdir()))
